In [1]:
# Install the required packages(required only once)
!pip install openai python-dotenv -q

In [18]:
import textwrap
# Loading the keys

import os
from dotenv import load_dotenv

import textwrap

def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)

# Load env file from the mentioned directory
load_dotenv('/home/furmani-arif/Documents/global-properties.env')

#fetch the openai api key fom the loaded env file
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with OPENAI_API_KEY"
    )

pretty_print("API key loaded successfully")



API key loaded successfully


In [19]:
# Load the OpenAI client
from openai import OpenAI

client = OpenAI(api_key = api_key)
MODEL = "gpt-5-nano"
pretty_print("OpenAI client is ready", "MODEL IS", MODEL)

OpenAI client is ready MODEL IS gpt-5-nano


In [20]:
response = client.responses.create(
    model=MODEL,
    input="What is the weather in Dhoraji right now? Can you fetch and let me know what's the exact temprature right now?",
    reasoning={"effort": "minimal"}
)
print(response.output_text)

I don’t have real-time weather access in this chat. To get the exact current temperature for Dhoraji right now, you can:

- Check a weather app or website (e.g., Weather.com, AccuWeather, BBC Weather) by searching “Dhoraji current temperature.”
- Use a voice assistant or default weather service on your device (siri, Google Assistant).
- If you share your preferred source, I can guide you how to look it up or interpret the forecast.

If you want, I can also provide typical climate info for Dhoraji (seasonal averages) or show you how to fetch live data from a weather API.


In [27]:
# Define math tools

add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

sub_tool = {
    "type": "function",
    "name": "subtract",
    "description": "Subtract two numbers and return the difference.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

mul_tool = {
    "type": "function",
    "name": "multiply",
    "description": "Multiply two numbers together and return the product.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

div_tool = {
    "type": "function",
    "name": "divide",
    "description": "Divide two numbers and return the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "Numerator (dividend)"},
            "b": {"type": "number", "description": "Denominator (divisor)"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        return "Error: Division by zero"
    return a / b
# how to tell the model what function exists, and what argument it takes. 

# Ask the model to add — but give it the tool
response = client.responses.create(
    model=MODEL,
    instructions="Use the tools available for any math. Never compute math yourself.",
    input="What is 7 + 12?",
    tools=[add_tool],
)

# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: add
  Arguments: {"a":7,"b":12}
  Call ID: call_WsWA3B1lL3N42JSPT4pHbfDt
 Response:


In [30]:
response = client.responses.create(
    model=MODEL,
    instructions="Use the tools for any and every kind of math. Never compute math yourself.",
    input = "I have right now fifteen oranges, 3.14 apple, 12.86 mangoes, but out of these 2.19 apples and 5.22 mangoes got rotten. How many fresh fruits do I have now?",
    tools=[add_tool, sub_tool, mul_tool, div_tool],
    reasoning={"effort": "low"}
)

print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: subtract
  Arguments: {"a":3.14,"b":2.19}
  Call ID: call_RNWPN989bp2ta30cJtpluYSh
  Type: function_call
  Function name: subtract
  Arguments: {"a":12.86,"b":5.22}
  Call ID: call_REfeC6KS1cKcQnYVRF13hcQC
 Response:


In [31]:
DISPATCH = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
}

In [34]:
import json
for item in response.output:
    if item.type == "function_call":
        func_name = item.name
        args = item.arguments
        call_id = item.call_id
        
        if func_name in DISPATCH:
            func = DISPATCH[func_name]
            mod_args = json.loads(args)  # Convert JSON string to Python dict
            result = func(**mod_args)  # Call the function with unpacked arguments
            print(f"Result of {func_name} with arguments {args}: {result}")
        else:
            print(f"Unknown function: {func_name}")

Result of subtract with arguments {"a":3.14,"b":2.19}: 0.9500000000000002
Result of subtract with arguments {"a":12.86,"b":5.22}: 7.64


In [ ]:
import requests

def get_weather(location):
    # Step 1: Get coordinates from city name
    geo_url = (
        f"https://geocoding-api.open-meteo.com/v1/search"
        f"?name={location}&count=1"
    )

    geo_response = requests.get(geo_url)
    geo_response.raise_for_status()

    geo_data = geo_response.json()

    if "results" not in geo_data:
        return f"City '{location}' not found"

    location = geo_data["results"][0]
    latitude = location["latitude"]
    longitude = location["longitude"]

    # Step 2: Fetch weather
    weather_url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={latitude}"
        f"&longitude={longitude}"
        "&current=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )

    weather_response = requests.get(weather_url)
    weather_response.raise_for_status()

    weather_data = weather_response.json()

    return {
        "city": location["name"],
        "country": location.get("country"),
        "temperature": weather_data["current"]["temperature_2m"],
        "humidity": weather_data["current"]["relative_humidity_2m"],
        "wind_speed": weather_data["current"]["wind_speed_10m"],
    }

In [ ]:
# Define a get_weather tool
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current temperature for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. 'Tel Aviv', 'London'"
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}

# Step 1: Ask with the weather tool
response = client.responses.create(
    model=MODEL,
    input="What's the weather like in Dhoraji and Surat?",
    tools=[weather_tool],
)

# Step 2 & 3: Find all function calls, execute each
print("Model requested these calls:")
new_input = list(response.output)

for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_weather(**args)
        print(f"  → {item.name}({args}) = {result}")
        
        # Step 4: Append our result
        new_input.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result,
            "output": json.dumps(result) if isinstance(result, dict) else str(result)
        })

# Step 5: Model composes final answer with real data
final = client.responses.create(
    model=MODEL,
    input=new_input,
    tools=[weather_tool],
)
print(f"\nFinal answer:\n{final.output_text}")

Model requested these calls:
  → get_weather({'location': 'Dhoraji'}) = {'city': 'Dhoraji', 'country': 'India', 'temperature': 37.8, 'humidity': 41, 'wind_speed': 24.6}
  → get_weather({'location': 'Surat'}) = {'city': 'Surat', 'country': 'India', 'temperature': 34.2, 'humidity': 59, 'wind_speed': 19.4}

Final answer:
Here’s the current weather:

- Dhoraji: 37.8°C, humidity 41%, wind 24.6 km/h
- Surat: 34.2°C, humidity 59%, wind 19.4 km/h

Both are quite hot right now; stay hydrated and use sun protection if you’re outdoors. Want me to fetch a forecast or set alerts?
